In [1]:
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import lsmr

In [2]:
INPUT_CSV = "players.csv"
OUTPUT_CSV = "AWS_players.csv"

@dataclass(frozen=True)
class AWSParams:
    psi_years: float = 2.0
    phi: float = 0.0062
    omega: float = 7.0

params = AWSParams()
params

AWSParams(psi_years=2.0, phi=0.0062, omega=7.0)

In [3]:
df = pd.read_csv(INPUT_CSV, parse_dates=["Match_Date"])
print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.head()

shape: (775378, 9)
columns: ['Unnamed: 0', 'Match_Date', 'Team', 'Team_ID', 'IsHome', 'Player', 'Player_ID', 'Player_rating', 'League']


,Unnamed: 0,Match_Date,Team,Team_ID,IsHome,Player,Player_ID,Player_rating,League
0,0,2016-05-15 15:00:00,Arsenal,13,1,Petr Cech,6775,7.08,England_Premier_League
1,1,2016-05-15 15:00:00,Arsenal,13,1,Héctor Bellerín,125211,8.38,England_Premier_League
2,2,2016-05-15 15:00:00,Arsenal,13,1,Gabriel Paulista,76810,7.51,England_Premier_League
3,3,2016-05-15 15:00:00,Arsenal,13,1,Laurent Koscielny,30051,8.35,England_Premier_League
4,4,2016-05-15 15:00:00,Arsenal,13,1,Nacho Monreal,23072,8.13,England_Premier_League


In [4]:
required = {"Match_Date", "League", "IsHome", "Player", "Player_ID", "Player_rating"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

summary = pd.Series({
    "date_min": df["Match_Date"].min(),
    "date_max": df["Match_Date"].max(),
    "rows": len(df),
    "players": df["Player_ID"].nunique(),
    "teams": df["Team_ID"].nunique() if "Team_ID" in df.columns else np.nan,
    "leagues": df["League"].nunique(),
    "missing_player_rating": int(df["Player_rating"].isna().sum()),
})
summary

date_min                 2015-08-07 19:30:00
date_max                 2026-05-24 20:45:00
rows                                  775378
players                                11658
teams                                    163
leagues                                    5
missing_player_rating                 208249
dtype: object

In [5]:
df["League"].value_counts(dropna=False)

League
Italy_Serie_A             171008
Spain_Laliga              169968
England_Premier_League    159096
France_Ligue1             146274
Germany_Bundesliga        129032
Name: count, dtype: int64

In [6]:
def clean_input(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(subset=["Player_rating", "Match_Date", "League", "Player_ID"]).copy()
    df["IsHome"] = df["IsHome"].astype(float)
    df["Player_ID"] = df["Player_ID"].astype(str)
    df["League"] = df["League"].astype(str)
    return df


def build_training_frame(df: pd.DataFrame, as_of_date: pd.Timestamp, params: AWSParams) -> pd.DataFrame:
    start_date = as_of_date - pd.Timedelta(days=int(round(params.psi_years * 365.25)))
    real = df[(df["Match_Date"] < as_of_date) & (df["Match_Date"] >= start_date)].copy()
    if real.empty:
        raise ValueError("No valid ratings in the selected lookback window.")

    days_old = (as_of_date - real["Match_Date"]).dt.total_seconds() / 86400.0
    real["sample_weight"] = np.exp(-params.phi * days_old / 3.5)
    real["is_pseudo"] = False

    latest = (
        real.sort_values("Match_Date")
        .groupby("Player_ID", as_index=False)
        .tail(1)[["Player_ID", "Player", "League"]]
    )
    league_avg = real.groupby("League")["Player_rating"].mean()
    global_avg = real["Player_rating"].mean()

    pseudo = latest.copy()
    pseudo["Match_Date"] = as_of_date
    pseudo["IsHome"] = 0.5
    pseudo["Player_rating"] = pseudo["League"].map(league_avg).fillna(global_avg).astype(float)
    pseudo["sample_weight"] = params.omega
    pseudo["is_pseudo"] = True

    cols = [
        "Match_Date", "League", "IsHome", "Player", "Player_ID",
        "Player_rating", "sample_weight", "is_pseudo",
    ]
    return pd.concat([real[cols], pseudo[cols]], ignore_index=True)

In [7]:
def fit_aws(df: pd.DataFrame, as_of_date, params: AWSParams):
    df = clean_input(df)
    as_of = pd.Timestamp(as_of_date) if as_of_date else df["Match_Date"].max() + pd.Timedelta(seconds=1)
    train = build_training_frame(df, as_of, params)

    player_codes, player_index = pd.factorize(train["Player_ID"], sort=True)
    league_codes, league_index = pd.factorize(train["League"], sort=True)
    n = len(train)
    n_players = len(player_index)
    n_leagues = len(league_index)

    intercept = sparse.csr_matrix(np.ones((n, 1)))
    home = sparse.csr_matrix(train[["IsHome"]].to_numpy(dtype=float))
    player_x = sparse.csr_matrix((np.ones(n), (np.arange(n), player_codes)), shape=(n, n_players))
    league_x = sparse.csr_matrix((np.ones(n), (np.arange(n), league_codes)), shape=(n, n_leagues))
    x = sparse.hstack([intercept, home, player_x, league_x], format="csr")

    sqrt_w = np.sqrt(train["sample_weight"].to_numpy(dtype=float))
    xw = x.multiply(sqrt_w[:, None])
    yw = train["Player_rating"].to_numpy(dtype=float) * sqrt_w
    coef = lsmr(xw, yw, atol=1e-10, btol=1e-10, maxiter=5000)[0]

    player_coef = coef[2 : 2 + n_players]
    league_coef = coef[2 + n_players :]

    names = (
        train.sort_values("Match_Date")
        .groupby("Player_ID", as_index=False)
        .tail(1)[["Player_ID", "Player", "League"]]
        .set_index("Player_ID")
    )
    real_only = train[~train["is_pseudo"]]
    appearances = real_only.groupby("Player_ID").size()
    weighted_apps = real_only.groupby("Player_ID")["sample_weight"].sum()
    rws = real_only.groupby("Player_ID")["Player_rating"].mean()

    result = pd.DataFrame({"Player_ID": player_index.astype(str), "Adjusted WhoScored Rating (AWS)": player_coef})
    result = result.join(names, on="Player_ID")
    result["Appearances in 2 year window"] = result["Player_ID"].map(appearances).fillna(0).astype(int)
    result["Weighted Appearances"] = result["Player_ID"].map(weighted_apps).fillna(0.0)
    result["Raw WhoScored Rating mean"] = result["Player_ID"].map(rws)
    result = result.sort_values("Adjusted WhoScored Rating (AWS)", ascending=False).reset_index(drop=True)

    meta = {
        "as_of_date": str(as_of),
        "real_observations": int((~train["is_pseudo"]).sum()),
        "pseudo_observations": int(train["is_pseudo"].sum()),
        "players": int(n_players),
        "leagues": int(n_leagues),
        "intercept": float(coef[0]),
        "home_advantage": float(coef[1]),
        "psi_years": params.psi_years,
        "phi": params.phi,
        "omega": params.omega,
    }
    for league, value in zip(league_index.astype(str), league_coef):
        meta[f"league_effect__{league}"] = float(value)

    return result, meta

In [8]:
# The End date of period
AS_OF_DATE = "2026-05-24"

aws_ratings, metadata = fit_aws(df, AS_OF_DATE, params)
metadata

{'as_of_date': '2026-05-24 00:00:00',
 'real_observations': 108687,
 'pseudo_observations': 3713,
 'players': 3713,
 'leagues': 5,
 'intercept': 5.4372879266511465,
 'home_advantage': 0.08014837266788404,
 'psi_years': 2.0,
 'phi': 0.0062,
 'omega': 7.0,
 'league_effect__England_Premier_League': 0.9799019808727113,
 'league_effect__France_Ligue1': 1.185242968642331,
 'league_effect__Germany_Bundesliga': 1.1367537138206132,
 'league_effect__Italy_Serie_A': 1.0732434117089735,
 'league_effect__Spain_Laliga': 1.0621458511223993}

In [9]:
aws_ratings.columns

Index(['Player_ID', 'Adjusted WhoScored Rating (AWS)', 'Player', 'League',
       'Appearances in 2 year window', 'Weighted Appearances',
       'Raw WhoScored Rating mean'],
      dtype='object')

In [10]:
aws_ratings[['Player', 'League', 'Appearances in 2 year window',
        'Adjusted WhoScored Rating (AWS)', 'Raw WhoScored Rating mean']].head(25)

,Player,League,Appearances in 2 year window,Adjusted WhoScored Rating (AWS),Raw WhoScored Rating mean
0,Lamine Yamal,Spain_Laliga,64,1.340278,8.093281
1,Harry Kane,Germany_Bundesliga,62,1.123296,7.927097
2,Michael Olise,Germany_Bundesliga,66,1.014278,7.790303
3,Kylian Mbappé,Spain_Laliga,65,0.966077,7.713692
4,Erling Haaland,England_Premier_League,66,0.862841,7.431970
5,Bruno Fernandes,England_Premier_League,70,0.820140,7.326286
6,Raphinha,Spain_Laliga,58,0.777416,7.535690
7,Nico Paz,Italy_Serie_A,70,0.724567,7.342143
8,Bukayo Saka,England_Premier_League,56,0.689756,7.302321
9,Jérémy Doku,England_Premier_League,58,0.671144,7.193793


In [11]:
aws_ratings.groupby("League")["AWS_rating"].describe()

KeyError: 'Column not found: AWS_rating'

In [ ]:
Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
aws_ratings.to_csv(OUTPUT_CSV, index=False)
print(OUTPUT_CSV)